# Luhya Whisper Small fine-tuning on Kaggle T4 x2

Before running: select **GPU T4 x2**, enable **Internet**, and add a Kaggle secret named `HF_TOKEN` with access to `Digital-Divide-Data/Luhya-ASR-Data-subset-40h`. W&B is not used. The notebook performs a two-step smoke test before spending resources on the full run.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-c", "import torch; print('CUDA devices:', torch.cuda.device_count()); assert torch.cuda.device_count() == 2, 'Select GPU T4 x2 in Kaggle settings'"], check=True)

In [ ]:
repo_dir = Path("/kaggle/working/luhya-asr")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/sam4rano/luhya-asr.git", str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Repository:", Path.cwd())

In [ ]:
# Keep Kaggle's CUDA-compatible PyTorch; install only the pinned ASR stack.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert os.environ["HF_TOKEN"], "HF_TOKEN is missing from Kaggle Secrets"
print("HF_TOKEN loaded; W&B disabled.")

## Resource-safety smoke test

This downloads/caches the dataset and model, checks both DDP processes, runs two optimizer steps, evaluates 24 examples, and writes to a separate `smoke-test` directory. The full run starts only if this succeeds.

In [ ]:
base_launch = [
    "accelerate", "launch", "--multi_gpu", "--num_processes", "2",
    "--mixed_precision", "fp16", "--num_cpu_threads_per_process", "2",
    "scripts/train_whisper.py",
    "--config", "config_files/ASR_train_config_whisper_small_kaggle.yaml",
]
subprocess.run(base_launch + ["--smoke_test"], check=True)

## Full fine-tuning

`latest` resumes a prior interrupted full run when a checkpoint exists; otherwise it starts from `openai/whisper-small`. Only one resumable checkpoint is retained.

In [ ]:
subprocess.run(base_launch + ["--resume_from_checkpoint", "latest"], check=True)

## Evaluation results

WER and CER below are measured on the retained validation and independent test splits. The JSON artifact also records whether existing splits were used or rebuilt and whether speaker overlap was found.

In [ ]:
import json
import pandas as pd
from IPython.display import display

results_dir = Path("/kaggle/working/luhya-asr-output/whisper-small-luhya")
metrics = pd.read_csv(results_dir / "metrics_summary.csv")
display(metrics.style.format({"wer_percent": "{:.2f}%", "cer_percent": "{:.2f}%"}))

with open(results_dir / "evaluation_summary.json", encoding="utf-8") as handle:
    summary = json.load(handle)
display(pd.DataFrame(summary["split_manifest"]["splits"]).T)
print("Split policy:", summary["split_manifest"]["split_policy"])
print("Speaker overlaps:", summary["split_manifest"]["speaker_overlap_counts"])
display(pd.read_csv(results_dir / "test_predictions.csv").head(20))

ax = metrics.set_index("split")[["wer_percent", "cer_percent"]].plot.bar(
    figsize=(7, 4), ylabel="Error rate (%)", rot=0, title="Luhya Whisper Small evaluation"
)
figure = ax.get_figure()
figure.tight_layout()
figure.savefig(results_dir / "evaluation_metrics.png", dpi=160, bbox_inches="tight")